## Encoding
### Opdracht

Zoals aan het begin van de cursus besproken gaat het project over handgeschreven nummers herekennen op een MysteryDevice met de volgende eigenschappen:

- Input scherm waarmee een nieuwe plaatje als ndarray aangemaakt kan worden door de gebruiker.
- Zeer beperkt RAM (256 KB)
- Beperkte opslag (1 MB voor 
programma + model)
- Geen GPU
- Embedded python


MysteryDevice opslaglimiet: 1 MB
Volledige MNIST dataset: 52 MB

Dus het opslaan van MNST voor training doeleinden zou niet werken. Maar in ons geval maakt het ook niet uit, want we kunnen onze AI methode op een PC trainen.

Echter, hebben we nog steeds een encoding van afbeeldingen nodig omdat de gebruiker wel een afbeelding invoert.

#### Hoe ziet de input eruit tijdens inference?

De gebruiker tekent op het touchscreen → dat moet naar 28×28 of een andere encoding worden gebracht.

#### Hoeveel RAM gebruikt die inputrepresentatie?

784 bytes (1 byte per pixel) -> heel laag
maar 32‑bits floats → 784 × 4 = 3 KB -> al een stuk hoger

Ontwerp een encoding van een MNIST plaatje die zo min mogelijk geheugen nodig heeft op het MysteryDevice om je Decision Tree uit de vorige opdracht te draaien op 1 sample. 

Mogelijk heb je al wat gedaan in de vorige stap, maar kijk of je het nog meer kan verbeteren.
Het moet:

- minder geheugen gebruiken
- minder rekenkracht nodig hebben
- toch informatie bewaren om cijfers te herkennen

Maak eventueel gebruik aan de volgende onderweren, maar ga vooral zelf experimenteren:
- binning
(“donker / medium / licht” → 3 waarden)

- binary thresholding
(zwart/wit → 1 bit per pixel)

- downscaling
(28×28 → 14×14 → 196 features)

- flattening
(784 → 784 vector)

- quantization
(8-bit → 4-bit → 2-bit per pixel)

- feature extraction
(bijv. “hoeveel inkt zit links/rechts/boven/onder?”)
- sparse encoding
(alleen niet‑nul pixels bewaren)

- Ga je normaliseren? Hoe?
- Ga je flattenen of 2D laten staan?
- Ga je binning toepassen op pixelwaarden?
- Ga je ordinale encoding toepassen op pixelintensiteiten?
- Ga je pixelwaarden reduceren (bijv. “donker, medium, licht”)?
- Ga je één pixel gebruiken, of alle 784?

**Ga ook een eigen “encoding” bedenken!**

In [1]:
from tensorflow.keras.datasets import mnist
import numpy as np

(X_train, y_train), _ = mnist.load_data()
sample = X_train[0]  # 28x28, uint8, waarden 0-255

print(f"Originele afbeelding: {sample.shape}, dtype={sample.dtype}, {sample.nbytes} bytes")
print(f"Label: {y_train[0]}")

Originele afbeelding: (28, 28), dtype=uint8, 784 bytes
Label: 5


In [2]:
# je kunt de volgende template gebruiken hier

def encode(img):
    # Stap 1: Downscale 28×28 → 14×14 door het maximum van elk 2×2 blok te nemen
    klein = img.reshape(14, 2, 14, 2).max(axis=(1, 3))
    # Stap 2: Binary thresholding — pixel boven 128 wordt 1 (inkt), de rest 0 (leeg)
    encoded = (klein > 128).astype(np.uint8)
    return encoded

encoded_sample = encode(sample)
encoded_sample

# bepaal hoeveel het gebruikt
encoded_sample.nbytes

196

Beantwoord ook de volgende vragen:

- Hoeveel RAM kost één afbeelding?
- Hoeveel RAM kost 100 afbeeldingen?
- Hoe groot zou het model maximaal mogen zijn?
- Kun je de encoding verder comprimeren?
- Is er informatie verloren gegaan?
- Kun je nog steeds het cijfer herkennen?

**Antwoorden op de ontwerpvragen:**

- **Normaliseren?** Nee — we gebruiken binary thresholding, geen normalisatie naar 0–1.
- **Flattenen of 2D?** We laten het 2D (14×14) — dat past bij de structuur van een afbeelding.
- **Binning?** Nee — we gebruiken binary thresholding (eenvoudiger dan binning).
- **Ordinale encoding?** Nee — we reduceren naar slechts 2 waarden: 0 (leeg) en 1 (inkt).
- **Pixelwaarden reduceren?** Ja — van 256 mogelijke waarden (uint8) naar 2 waarden (0 of 1).
- **Hoeveel pixels?** Alle 196 pixels (na downscaling van 784 naar 196).

**Eigen encoding:** Downscaling (28×28 → 14×14) gecombineerd met Binary Thresholding.
We pakken het maximale pixel in elk 2×2 blok (zodat dunne lijnen niet verdwijnen) en zetten daarna alles naar 0 of 1.

In [3]:
een_afbeelding     = encoded_sample.nbytes
honderd            = een_afbeelding * 100
ram_budget         = 256 * 1024
model_budget       = ram_budget - een_afbeelding

print(f"RAM voor één afbeelding:   {een_afbeelding} bytes")
print(f"RAM voor 100 afbeeldingen: {honderd} bytes ({honderd / 1024:.1f} KB)")
print(f"RAM-budget (256 KB):       {ram_budget} bytes")
print(f"Ruimte over voor model:    {model_budget} bytes ({model_budget / 1024:.1f} KB)")
print()

# Verder comprimeren met packbits (1 bit per pixel in plaats van 1 byte)
packed = np.packbits(encoded_sample)
print(f"Verder comprimeren? Ja — met numpy packbits: {packed.nbytes} bytes (was {een_afbeelding} bytes)")
print()
print("Informatie verloren? Ja — grijstinten zijn weg, alleen zwart/wit.")
print("Nog herkenbaar?     Ja — de vorm van het cijfer blijft zichtbaar.")

RAM voor één afbeelding:   196 bytes
RAM voor 100 afbeeldingen: 19600 bytes (19.1 KB)
RAM-budget (256 KB):       262144 bytes
Ruimte over voor model:    261948 bytes (255.8 KB)

Verder comprimeren? Ja — met numpy packbits: 25 bytes (was 196 bytes)

Informatie verloren? Ja — grijstinten zijn weg, alleen zwart/wit.
Nog herkenbaar?     Ja — de vorm van het cijfer blijft zichtbaar.
